# **01: Data Preprocessing**
---

## **Table of Contents**
1. [Objective](#obj)
2. [Dataset Info](#data-info)
3. [Summary](#summary)
4. [Import Libraries](#import)
5. [Initial Inspection](#inspect)
6. [Drop Irrelevant/Unused Columns](#drop-col)
7. [Outlier Detection Strategy](#out-detect)
8. [Outlier Handling](#out-handl)
9. [Feature Aggregation](#feat-agg)
---

<a id ='obj'></a>
## **Objective**
To clean, format, and reduce the dataset dimensionality by resolving multicollinearity, structuring time-series metadata, and preserving natural usage fluctuations for predictive modeling.

<a id ='data-info'></a>
## **Dataset Info**
* **Source:** UCI Machine Learning Repository
* **Title:** Appliances Energy Prediction
* **Time Range:** 11th January 2016 - 27th May 2016
* **Primary Target Variable:** `Appliances` (Watt-hour consumed)

<a id ='summary'></a>
## **Summary**
In this notebook, here's the accomplished decision:
>  ####  1. Convert 'date' DataType to datetime64 instead of string
> **Justification**: Easier management and future usage, such as to subscript by time features (hour, day, month etc.).

>  ####  2. Set 'date' as index
> **Justification**: Easier management, suitable for time-series analysis.

> #### 3. Drop Columns Representing Readings from Cheivres Airport
> **Justification**: Data recorded from miles away introduces undesired spatial lag. Furthemore, we already have an outdoor temperature and humidity reading from the wall sensor (T6, RH_6).

> #### 4. No outlier handling done
> **Justification**: All values are within the plausible range. Furthermore, fluctuations exist, but are consistent throughout the time range and no prominent irregularities found. Hence, it is kept the way it is to preserve information about the natural fluctuation of electrical consumption.

> #### 5. Aggregate Temperature and Humidity Features by Location
> The grouping is as follows:
> 1. Main Indoor Areas
> **Location**: kitchen, laundry room, ironing room, office room, teenager room, parents room
> 2. Living Room
> **Location**: living room
> 3. Bathroom
> **Location**: bathroom
> 4. Outdoor
> **Location**: outside of the building (north side)
>
> **Justification**: Eliminates multicollinearity as heat and moisture diffuse into adjacent room. Locations with high correlation across both temperature and humidity is averaged together, and vice versa.





<a id ='import'></a>
## **Import Libraries**

In [ ]:
# Import relevant packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os

<a id ='inspect'></a>
## **Initial Inspection**
This section will inspect the dataset structure, statistics, characteristics and quality.

### Import dataset into pandas DataFrame

In [ ]:
# Get parent address
PARENT_ADDRESS = Path(os.getcwd()).parent

# Import csv file
energy = pd.read_csv(f'{PARENT_ADDRESS}/data/raw/energydata_complete.csv')

### Inspect samples

In [ ]:
energy.head(3)

### Inspect dataset characteristics

In [ ]:
# Print dataset info
energy.info()

### Check for Duplicated Index (date)

In [ ]:
# Check for duplicated date, as this will serve as index
print(f'Number of Duplicated Date: {energy['date'].duplicated().sum()}')

> ### 💡 **Key Takeaways**
> 1. **```No missing values found```**
>
> 2. **```Date column have unconventional data type```**
> 
>       **Decision**: Convert it to 'datetime64'and set as index.
>
> 3. **```No duplicated index```**

### Decision execution

In [ ]:
# Convert 'date' into datetime64 format
energy['date'] = energy['date'].astype('datetime64[s]')

# Set 'date' as index
energy = energy.set_index('date')

### Inspect changes

In [ ]:
energy.info()

In [ ]:
energy.describe()

<a id ='drop-col'></a>
## **Drop Irrelevant/Unused Features**
We'll drop the features related to the readings taken from Chievres Airport, as data recorded from miles away introduces undesired spatial lag. Furthemore, we already have an outdoor temperature and humidity reading from the wall sensor (T6, RH_6).

In [ ]:
# Define unused columns
unused = [
    'T_out',
    'Press_mm_hg',
    'RH_out',
    'Windspeed',
    'Visibility',
    'Tdewpoint'
]

# Drop columns
energy = energy.drop(unused, axis = 1)

In [ ]:
# Check modified dataset
energy.dtypes

<a id ='out-detect'></a>
## **Outlier Detection Strategy**

> ### 1. Separate Features
For this part, we'll separate the dataset into 3 parts:
1. **Target feature** (Appliances)
2. **Temperature Feature** (T1 - T9)
3. **Humidity Feature** (RH_1 - RH_9)

**Justification**: Different scales, so features with smaller range and overall values tend to be flattened when plotted. For example, plotting Appliances (range ~1070 Wh) with temperature (range ~15 °C)  and humidity (range ~100 % )features in a time-series graph will flatten temperature the most. This will make visual inspection to detect spikes in temperature features much less prominent.

> ### 2. Aggregate by Rolling Mean
**Justification**: The dataset is sampled once every 10 minutes, which results in 144 samples each day. Plotting these raw values will result in a very noisy time-series plot.

> ### 3. 12-Hour as Window Size
**Justification**: The data was sampled in the course of 4 months, which is not long enough to reflect major seasonality change (winter, summer etc.). Hence, a 12-hour (or 72 samples) rolling window size is sufficient to reflect daily seasonality change (day and night) only.

### Visual Inspection

In [ ]:
grouping = {
    "target": ["Appliances"],
    "temp": [f"T{i + 1}" for i in range(9)],
    "humid": [f"RH_{i + 1}" for i in range(9)],
}

# Create canvas
fig, axs = plt.subplots(nrows=len(grouping), ncols=1, figsize=(14, 10))

# Plot by group
for i, (group_name, columns) in enumerate(grouping.items()):
    for col in columns:
        energy[col].rolling(window=72).mean().plot(ax=axs[i], label=col)

    axs[i].set_title(f"12-Hour Rolling Average: {group_name.upper()}")
    axs[i].set_ylabel("Value")
    axs[i].legend(loc="upper right", bbox_to_anchor=(1.12, 1), fontsize=8)

plt.tight_layout()
plt.show()

> ### 💡 **Key Takeaways**
> 1. **```No Illogical Values```**
> 
>> Upon inspection, all values fall within a plausible range, hence we can assume no outliers are caused by sensor error or key-in mistakes.
>
> 2. **```Fluctuations Are Consistent, Not Irregular```**
>
>> No fluctuations stands out prominently among the rest of the spikes. This means that the fluctuations reflects true variation caused by real usage.


<a id ='out-handl'></a>
## **Outlier Handling Strategy**
> ### Leave it the Way It Is
**Justification**: As all values seem to reflect natural variation of real-life electrical usage with no prominent irregularities, we'll leave it as it is to preserve the information.

<a id ='feat-agg'></a>
## **Feature Aggregation**
Currently, the dataset contains 28 attributes, which can be complex to analyze. Furthermore, heat and moisture diffuse across adjacent room, which means that placing 28 sensors in a single house might be excessive and expensive. Hence, feature aggregation seems appropriate here.

We'll first look at feature collinearity to decide on the proper grouping. For visibility purposes, we'll divide the heatmap by temperature and humidity features.

### Determine Grouping Strategy

In [ ]:
# Define location sequence
loc = [
    "kitchen",
    "living room",
    "laundry room",
    "office room",
    "bathroom",
    "outside",
    "ironing room",
    "teenager room",
    "parents room",
]

# Initialize column names T1 - T9 and RH_1 - RH_9
temp_feature = [f"T{i+1}" for i in range(9)]
humid_feature = [f"RH_{i+1}" for i in range(9)]

# Calculate correlations
temp_corr = energy[temp_feature].corr(method="pearson")
humid_corr = energy[humid_feature].corr(method="pearson")

# Temperature Features
# ---
cg_temp = sns.clustermap(
    temp_corr,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    annot=False,
    linewidths=0.5,
    figsize=(4, 4),
    cbar_pos=(1.2, 0.2, 0.03, 0.4),
)

# Reorder locations based on temperature dendrogram
temp_labels = [loc[i] for i in cg_temp.dendrogram_col.reordered_ind]

# Set X and Y labels
cg_temp.ax_heatmap.set_xticklabels(temp_labels, rotation=90)
cg_temp.ax_heatmap.set_yticklabels(temp_labels, rotation=0)

# Set title
cg_temp.fig.suptitle(
    "Temperature Feature Correlation by Location", y=1.02, fontsize=12
)


# Humidity Features
# ---
cg_humid = sns.clustermap(
    humid_corr,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    annot=False,
    linewidths=0.5,
    figsize=(4, 4),
    cbar_pos=(1.2, 0.2, 0.03, 0.4),
)

# Reorder locations based on humidity dendrogram
humid_labels = [loc[i] for i in cg_humid.dendrogram_col.reordered_ind]

# Set X and Y labels
cg_humid.ax_heatmap.set_xticklabels(humid_labels, rotation=90)
cg_humid.ax_heatmap.set_yticklabels(humid_labels, rotation=0)

# Set title
cg_humid.fig.suptitle(
    "Humidity Feature Correlation by Location", y=1.02, fontsize=12
)

# Render both figures
plt.tight_layout()
plt.show()

Based on the heatmaps, this grouping seems appropriate:

### ```1. Main Indoor Areas``` 
> **Location**: kitchen, laundry room, ironing room, office room, teenager room, parents room
>
> **Justification**: High correlation across both temperature and humidity (signified by deep red color). Averaging eliminates severe multicollinearity with negligible loss of signal.

### ```2. Living Room```
> **Location**: Living room only
>
> **Justification**: Temperature transition similar towards outdoor condition

### ```3. Bathroom```
> **Location**: Bathroom only
>
> **Justification**: Humidity forms an isolated cluster with low overall correlation (signified by pale red color) to all other indoor rooms.

### ```4. Outdoor```
> **Location**: Outside only
>
> **Justification**: Low correlation with every other location, either in terms of temperature or humidity.

### Apply Grouping Using Average

In [ ]:
# Define the grouping using number codes taken from UCI Appliances Energy page
# For reference: https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction
groups = {
    "indoor": [1, 3, 4, 7, 8, 9],
    "liv_room": [2],
    "bathroom": [5],
    "outdoor": [6],
}

for name, indices in groups.items():
    # Mean Temperature
    temp_cols = [f"T{i}" for i in indices]
    energy[f"T_{name}"] = energy[temp_cols].mean(axis=1)

    # Mean Humidity
    rh_cols = [f"RH_{i}" for i in indices]
    energy[f"RH_{name}"] = energy[rh_cols].mean(axis=1)

In [ ]:
# Drop old, unaggregated features
for i in range (9):
    energy = energy.drop([f'T{i+1}',  f'RH_{i+1}'], axis = 1)

In [ ]:
# Visualize changes
energy.head(3)